# Astro spectroscopy webb app

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
# Load the dataset
df_galaxies = pd.read_csv(
    "/home/catherine-elizabeth-halsey/Documents/github/sprint-7-bpt/astro-spectroscopy-visualization/sdss_galaxies.csv"
)

In [3]:
df_galaxies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140035 entries, 0 to 140034
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   plate               140035 non-null  int64  
 1   fiberid             140035 non-null  int64  
 2   mjd                 140035 non-null  int64  
 3   z                   140035 non-null  float64
 4   zwarning            140035 non-null  int64  
 5   h_alpha_flux        140035 non-null  float64
 6   h_alpha_flux_err    140035 non-null  float64
 7   h_beta_flux         140035 non-null  float64
 8   h_beta_flux_err     140035 non-null  float64
 9   nii_6584_flux       140035 non-null  float64
 10  nii_6584_flux_err   140035 non-null  float64
 11  oiii_5007_flux      140035 non-null  float64
 12  oiii_5007_flux_err  140035 non-null  float64
dtypes: float64(9), int64(4)
memory usage: 13.9 MB


In [4]:
df_galaxies.columns

Index(['plate', 'fiberid', 'mjd', 'z', 'zwarning', 'h_alpha_flux',
       'h_alpha_flux_err', 'h_beta_flux', 'h_beta_flux_err', 'nii_6584_flux',
       'nii_6584_flux_err', 'oiii_5007_flux', 'oiii_5007_flux_err'],
      dtype='object')

In [5]:
df_galaxies.head()

,plate,fiberid,mjd,z,zwarning,h_alpha_flux,h_alpha_flux_err,h_beta_flux,h_beta_flux_err,nii_6584_flux,nii_6584_flux_err,oiii_5007_flux,oiii_5007_flux_err
0,2115,42,53535,0.057162,0,119.6482,2.363719,33.81229,2.261494,35.45956,1.643917,10.77501,2.107138
1,906,315,52368,0.046144,0,151.8282,3.690749,25.73299,2.673304,66.15668,2.617815,17.35538,2.424217
2,2022,473,53827,0.040476,0,1454.7450,13.445210,323.17360,5.726914,576.86880,6.294590,98.13634,4.082010
3,2031,600,53848,0.034786,0,256.8179,3.622716,54.58832,2.016292,85.42187,2.145393,26.48615,1.869704
4,2016,477,53799,0.068709,0,183.2177,3.994831,38.52705,2.857010,80.47491,3.183102,44.21381,3.210339


In [6]:
# Emission line fluxes
nii  = df_galaxies["nii_6584_flux"]
ha   = df_galaxies["h_alpha_flux"]
oiii = df_galaxies["oiii_5007_flux"]
hb   = df_galaxies["h_beta_flux"]

# Ratios used in BPT
ratio_x = nii / ha
ratio_y = oiii / hb

# Remove invalid / non-positive values before log10
mask = (
    np.isfinite(ratio_x) & (ratio_x > 0) &
    np.isfinite(ratio_y) & (ratio_y > 0)
)

plot_df_galaxies = pd.DataFrame({
    "log_NII_Ha": np.log10(ratio_x[mask]),
    "log_OIII_Hb": np.log10(ratio_y[mask])
})

In [7]:
# 0-count bins stay white; density fades toward near-white peak
inferno_pink_fade = [
    [0.00, "white"],

    [0.02, "#ff007f"],
    [0.06, "#ff1a8c"],
    [0.10, "#ff3399"],
    [0.14, "#ff4da6"],
    [0.18, "#ff66b3"],
    [0.22, "#ff80bf"],
    [0.26, "#ff99cc"],
    [0.30, "#ffb3d9"],

    [0.36, "#ffc2d1"],
    [0.42, "#ffd1c9"],
    [0.48, "#ffdfbf"],
    [0.54, "#ffeab3"],

    [0.60, "#fff2a6"],
    [0.66, "#fff599"],
    [0.72, "#fff88c"],
    [0.78, "#fffbb3"],

    [0.84, "#fffdd0"],
    [0.88, "#fffde0"],
    [0.92, "#fffef0"],
    [0.95, "#fffef7"],
    [0.97, "#fffefc"],
    [0.99, "#ffffff"],
    [1.00, "#fffffb"]
]

In [8]:
fig = px.density_heatmap(
    plot_df_galaxies,
    x="log_NII_Ha",
    y="log_OIII_Hb",
    nbinsx=700,   # controls square size
    nbinsy=700,
    title="BPT Diagram (SDSS) — Density"
)

fig.update_layout(
    template="simple_white",
    width=900,
    height=700,
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis=dict(
        colorscale=inferno_pink_fade,
        cmin=1,  # ensures 0-density = white
        colorbar=dict(title="counts")
    )
)

# Paper-style axes
fig.update_xaxes(
    range=[-2, 0.5],
    title="log([NII] 6584 / Hα)",
    showline=True,
    linecolor="black",
    ticks="outside",
    tickfont=dict(color="black", size=12),
    showgrid=True,
    gridcolor="rgba(0,0,0,0.10)"
)

fig.update_yaxes(
    range=[-1, 1.5],
    title="log([OIII] 5007 / Hβ)",
    showline=True,
    linecolor="black",
    ticks="outside",
    tickfont=dict(color="black", size=12),
    showgrid=True,
    gridcolor="rgba(0,0,0,0.10)"
)

fig.show()

In [ ]:
"""
# --- Load SDSS CSV ---
df = pd.read_csv(
    "sdss_galaxies.csv"
)

nii = df["nii_6584_flux"]
ha = df["h_alpha_flux"]
oiii = df["oiii_5007_flux"]
hb = df["h_beta_flux"]

ratio_x = nii / ha
ratio_y = oiii / hb

mask = (
    np.isfinite(ratio_x) & (ratio_x > 0) &
    np.isfinite(ratio_y) & (ratio_y > 0)
)

x = np.log10(ratio_x[mask])
y = np.log10(ratio_y[mask])

plot_df = pd.DataFrame({
    "log_NII_Ha": x,
    "log_OIII_Hb": y
})

# --- density heatmap ---
fig = px.density_heatmap(
    plot_df,
    x="log_NII_Ha",
    y="log_OIII_Hb",
    nbinsx=700,
    nbinsy=700,
    title="BPT Diagram (SDSS) — Density"
)

# Inferno-ish base -> hot pink -> near-white peak
inferno_pink_fade = [
    [0.00, "white"],

    # rosa fuerte inicial
    [0.02, "#ff007f"],
    [0.06, "#ff1a8c"],
    [0.10, "#ff3399"],
    [0.14, "#ff4da6"],
    [0.18, "#ff66b3"],
    [0.22, "#ff80bf"],
    [0.26, "#ff99cc"],
    [0.30, "#ffb3d9"],

    # transición rosa → durazno
    [0.36, "#ffc2d1"],
    [0.42, "#ffd1c9"],
    [0.48, "#ffdfbf"],
    [0.54, "#ffeab3"],

    # durazno → amarillo claro
    [0.60, "#fff2a6"],
    [0.66, "#fff599"],
    [0.72, "#fff88c"],
    [0.78, "#fffbb3"],

    # amarillo → blanco
    [0.84, "#fffdd0"],
    [0.88, "#fffde0"],
    [0.92, "#fffef0"],
    [0.95, "#fffef7"],
    [0.97, "#fffefc"],
    [0.99, "#ffffff"],

    # pico brillante
    [1.00, "#fffffb"]
]

fig.update_layout(
    template="simple_white",
    width=900,
    height=700,
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis=dict(
        colorscale=inferno_pink_fade,
        cmin=1,  # makes 0-count bins show as white
        colorbar=dict(title="counts")
    )
)

fig.update_xaxes(
    range=[-2, 0.5],
    title="log([NII] 6584 / Hα)",
    showline=True,
    linecolor="black",
    ticks="outside",
    tickfont=dict(color="black", size=12),
    showgrid=True,
    gridcolor="rgba(0,0,0,0.10)"
)

fig.update_yaxes(
    range=[-1, 1.5],
    title="log([OIII] 5007 / Hβ)",
    showline=True,
    linecolor="black",
    ticks="outside",
    tickfont=dict(color="black", size=12),
    showgrid=True,
    gridcolor="rgba(0,0,0,0.10)"
)

fig.show()
"""